# DUMPLINGs Colab Assistant Journal Control Panel

A lightweight Colab workflow for rebuilding factual indices and running the assistant journals on copied `runs/` artifacts.

This notebook is intentionally separate from the training notebook. It stages only the light analysis workspace into `/content`.

In [ ]:
from pathlib import Path

DRIVE_REPO_ROOT = "/content/drive/MyDrive/DUMPLINGs"
COLAB_WORKSPACE = "/content/DUMPLINGs_analysis"

RUN_PREFIXES = [
    "DUMPLING_A1_esm_only",
    "DUMPLING_A1_dimenet_only",
    "DUMPLING_A1_full",
    "DUMPLING_A1_dimenet_esm"
]

ASSISTANT_MODE = "dry-run"  # "dry-run" or "live"
ASSISTANT_MODEL = "qwen2.5:7b"
ASSISTANT_LIMIT = 0       # 0 means all copied runs
FORCE_REFRESH = False
OLLAMA_TIMEOUT_SEC = 1800

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    f"{DRIVE_REPO_ROOT}/scripts/colab_stage_analysis_workspace.py",
    "--src", DRIVE_REPO_ROOT,
    "--dst", COLAB_WORKSPACE,
]
for prefix in RUN_PREFIXES:
    cmd.extend(["--include-run-prefix", prefix])

subprocess.run(cmd, check=True)
print(f"Staged analysis workspace into {COLAB_WORKSPACE}")

In [ ]:
%cd /content/DUMPLINGs_analysis
!python3 scripts/rebuild_experiment_index.py --runs-dir runs

In [ ]:
import os
import subprocess
import time
import urllib.request
from pathlib import Path

if ASSISTANT_MODE == "live":
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    subprocess.Popen(
        "OLLAMA_HOST=127.0.0.1:11434 ollama serve > /tmp/ollama.log 2>&1",
        shell=True,
    )
    health_url = "http://127.0.0.1:11434/api/tags"
    for _ in range(60):
        try:
            with urllib.request.urlopen(health_url, timeout=3):
                break
        except Exception:
            time.sleep(2)
    else:
        raise RuntimeError("Ollama did not become ready in time")

    subprocess.run(["ollama", "pull", ASSISTANT_MODEL], check=True)
    env_path = Path("assistant/.env")
    env_path.write_text(
        "\n".join([
            "ASSISTANT_LLM_PROVIDER=ollama",
            f"ASSISTANT_LLM_MODEL={ASSISTANT_MODEL}",
            f"ASSISTANT_LLM_TIMEOUT_SEC={OLLAMA_TIMEOUT_SEC}",
            "ASSISTANT_LLM_TEMPERATURE=0.2",
        ]) + "\n",
        encoding="utf-8",
    )
    print(env_path.read_text(encoding="utf-8"))
else:
    print("Dry-run mode selected; skipping Ollama install and model pull.")

In [ ]:
import subprocess

cmd = ["bash", "assistant/run_llm_journal.sh", f"--{ASSISTANT_MODE}"]
if ASSISTANT_LIMIT > 0:
    cmd.extend(["--limit", str(ASSISTANT_LIMIT)])
if FORCE_REFRESH:
    cmd.append("--force-refresh")

subprocess.run(cmd, check=True)

In [ ]:
from pathlib import Path

for rel_path in [
    "runs/experiment_journal_llm.md",
    "runs/experiment_series_journal_llm.md",
]:
    path = Path(rel_path)
    print(f"\n===== {rel_path} =====\n")
    if path.exists():
        print(path.read_text(encoding="utf-8")[:12000])
    else:
        print("missing")

## Notes

- This notebook is happy with copied `runs/` folders; it does not need the training stack.
- If you want to preserve the generated LLM journals back to Drive, copy the files from `/content/DUMPLINGs_analysis/runs/` into your Drive repo after inspection.
- If Colab GPU memory is tight, try a smaller Ollama model tag before concluding the workflow is bad.